# 10 – Interactive queries with Trino

**Goal (Task 5, parts 1–3):**
1. A Trino **cluster**: 1 coordinator + 2 workers, with the **Trino UI** → list the **nodes**; two **catalogs** (database + pile) → list the **catalogs**.
2. Queries on the **database** and on the **pile**, an **older snapshot**, and **partition pruning**.
3. A **federated query** that combines the database and the pile in one SQL statement, and which part runs **in PostgreSQL** (pushdown) and which **in Trino**.

**Trino** is a distributed SQL engine. It stores no data itself: it **queries data where it lives** (PostgreSQL, Iceberg tables in Silo, …) through **catalogs**, and can combine them in one query.

| Container | Role |
|---|---|
| `trino-coordinator` | receives SQL, plans it, splits it into tasks, collects the result, hosts the web UI (http://localhost:8080/ui/) |
| `trino-worker-1`, `trino-worker-2` | execute the tasks: read data, filter, join, aggregate |

In [ ]:
import pandas as pd
from trino_pile import sql, run, explain, query_stats, wait_for_cluster, TRINO_UI

pd.set_option("display.max_colwidth", 200)
print("Trino UI:", TRINO_UI, "(log in with any username, e.g. pawhome)")

## Part 1 – The cluster

### Nodes

`system.runtime.nodes` lists every node that registered with the coordinator. The cell waits until all 3 are **active** (Trino needs ~30–60 s to start).

In [ ]:
nodes = wait_for_cluster(expected_nodes=3)
nodes

- `trino-coordinator` → `coordinator = True`. With `node-scheduler.include-coordinator=false` it only plans; it does no data work.
- `trino-worker-1`, `trino-worker-2` → `coordinator = False`: they do the work.

The same list is in the **Trino UI** (http://localhost:8080/ui/): "Active workers: 2", plus every query with its stages and splits.

### Catalogs

A **catalog** = one configured data source (a file in `trino/catalog/`):

| Catalog | File | Connector | What it reads |
|---|---|---|---|
| `pawhome_db` | `pawhome_db.properties` | `postgresql` | the operational database (Task 1) |
| `pile` | `pile.properties` | `iceberg` (JDBC catalog = the same SQLite file as PyIceberg) | the Iceberg tables in Silo (Tasks 3–4) |
| `system` | built in | | information about Trino itself (nodes, queries) |

In [ ]:
sql("SHOW CATALOGS")

In [ ]:
display(sql("SHOW SCHEMAS FROM pawhome_db"))
display(sql("SHOW SCHEMAS FROM pile"))       # = the medallion zones
sql("SHOW TABLES FROM pile.gold")

## Part 2 – Queries on the database and on the pile

### A query on the operational database (PostgreSQL)

In [ ]:
sql("""
    SELECT species, status, count(*) AS animals
    FROM pawhome_db.public.animals
    GROUP BY species, status
    ORDER BY species, status
""")

### A query on the pile (Iceberg table)

In [ ]:
sql("""
    SELECT species, count(*) AS animals, round(avg(age_years), 1) AS avg_age,
           sum(CASE WHEN found_home THEN 1 ELSE 0 END) AS found_home
    FROM pile.silver.animals
    GROUP BY species
    ORDER BY animals DESC
""")

### Reading an older snapshot (time travel)

Every Iceberg table has hidden **metadata tables**, like `"animals$snapshots"`. With `FOR VERSION AS OF <snapshot_id>` Trino reads the table **as it was** at that snapshot.

In [ ]:
snapshots = sql("""
    SELECT snapshot_id, committed_at, operation,
           element_at(summary, 'added-records') AS added_rows, element_at(summary, 'total-records') AS total_rows
    FROM pile.silver."animals$snapshots"
    ORDER BY committed_at
""")
snapshots

In [ ]:
first_snapshot = int(snapshots.snapshot_id.iloc[0])

pd.DataFrame({
    "version": [f"OLD (snapshot {first_snapshot})", "CURRENT"],
    "animals": [sql(f"SELECT count(*) FROM pile.silver.animals FOR VERSION AS OF {first_snapshot}").iloc[0, 0],
                sql("SELECT count(*) FROM pile.silver.animals").iloc[0, 0]],
})

In [ ]:
# Rows that are different between the old snapshot and now (new or changed animals)
sql(f"""
    SELECT id, name, species, breed FROM pile.silver.animals
    EXCEPT
    SELECT id, name, species, breed FROM pile.silver.animals FOR VERSION AS OF {first_snapshot}
    ORDER BY id
""")

### Partition pruning

`gold.shelter_monthly_activity` is partitioned by **month** (Task 4). `SHOW CREATE TABLE` shows the partition spec:

In [ ]:
print(sql("SHOW CREATE TABLE pile.gold.shelter_monthly_activity").iloc[0, 0])

In [ ]:
# One partition (= one month) per row, with the number of data files in it
sql("""
    SELECT partition, record_count, file_count
    FROM pile.gold."shelter_monthly_activity$partitions"
    ORDER BY partition DESC
""").head(6)

Now the **same aggregation twice**: once over the whole table, once with a **filter on the partition column** (one month). Trino's statistics show how much data each query really read.

In [ ]:
last_month = sql("SELECT max(month_start) FROM pile.gold.shelter_monthly_activity").iloc[0, 0]
month = (pd.Timestamp(last_month) - pd.offsets.MonthBegin(1)).date()     # last FULL month

full_result, full_stats = run("""
    SELECT sum(applications) AS applications, sum(adoptions) AS adoptions
    FROM pile.gold.shelter_monthly_activity
""")
month_result, month_stats = run(f"""
    SELECT sum(applications) AS applications, sum(adoptions) AS adoptions
    FROM pile.gold.shelter_monthly_activity
    WHERE month_start >= DATE '{month}' AND month_start < DATE '{month}' + INTERVAL '1' MONTH
""")

pd.DataFrame({"whole table": query_stats(full_stats), f"only {month:%B %Y}": query_stats(month_stats)})

**What to notice:** with the partition filter Trino read only the files of **one month** (far fewer rows, bytes and splits). Iceberg's metadata says which month every file holds, so the other files are **skipped without opening them**: that's **partition pruning**. Open the "Trino UI" link of each query to see the same numbers.

In [ ]:
# The plan: the filter on month_start is handed to the Iceberg table scan (constraint on [month_start])
explain(f"""
    SELECT sum(applications) FROM pile.gold.shelter_monthly_activity
    WHERE month_start >= DATE '{month}' AND month_start < DATE '{month}' + INTERVAL '1' MONTH
""", keep=["TableScan", "constraint", "month_start"]);

In [ ]:
# The same idea on the second gold table, partitioned by species
_, all_species = run("SELECT sum(animals) FROM pile.gold.breed_popularity")
_, only_dogs = run("SELECT sum(animals) FROM pile.gold.breed_popularity WHERE species = 'dog'")
pd.DataFrame({"all species": query_stats(all_species), "species = 'dog'": query_stats(only_dogs)})

## Part 3 – A federated query (database + pile in one SQL statement)

**Business question:** *Which animals are **still available right now**, have the **lowest predicted chance** of finding a home, and how many **pending applications** do they have today?* → the shelter should promote these animals first.

- the **prediction** comes from the **pile** (`pile.gold.found_home_predictions`, Task 4 machine learning)
- the **current status** and the **pending applications** come **live from the database** (`pawhome_db.public.animals`, `applications`)

In [ ]:
FEDERATED = """
SELECT p.id, p.name, p.species, p.prob_found_home,
       a.status AS status_now,
       coalesce(l.pending_applications, 0) AS pending_applications_now
FROM pile.gold.found_home_predictions AS p                         -- from the PILE (Iceberg)
JOIN pawhome_db.public.animals AS a                                -- LIVE from PostgreSQL
  ON a.id = p.id
LEFT JOIN (
    SELECT animal_id, count(*) AS pending_applications             -- LIVE from PostgreSQL
    FROM pawhome_db.public.applications
    WHERE status = 'pending'
    GROUP BY animal_id
) AS l ON l.animal_id = p.id
WHERE a.status = 'available'
ORDER BY p.prob_found_home ASC
LIMIT 10
"""
sql(FEDERATED)

### What runs in PostgreSQL and what runs in Trino?

`EXPLAIN` shows the plan. Look at the **table scans** of `pawhome_db`:
- **Predicate pushdown** (filters sent to the database): `constraint on [status]` / `WHERE status = 'pending'`, so PostgreSQL returns only matching rows.
- **Projection pushdown** (only needed columns): the scan lists only the few columns used (e.g. `id`, `status`), not all 12 columns of `animals`.
- **Aggregation pushdown**: the `count(*) … GROUP BY animal_id` appears **inside** the scan as `Query[SELECT … GROUP BY …]`, so PostgreSQL counts.

Everything **above** the scans runs **in Trino** (on the workers): reading the Iceberg file of `found_home_predictions`, the **joins** between the two sources (data from two different systems can only be joined in Trino), **ORDER BY … LIMIT** (`TopN`) and the final output.

In [ ]:
plan = explain(FEDERATED, keep=["TableScan", "Query[", "constraint", "pawhome_db", "pile:", "Join", "TopN", "Aggregate"])

In [ ]:
print(plan)      # the full plan, if you want to see everything

In [ ]:
# A second, simple example: the WHOLE aggregation is pushed down to PostgreSQL
explain("""
    SELECT status, count(*) FROM pawhome_db.public.applications GROUP BY status
""", keep=["TableScan", "Query["]);